## OBJECTIVE

Analyze the data to know what are the most common words said during the elections, whether those are good or bad words to the candidates and how their names appeared in the comments of the people. The tendency, whether they grew or decrease with time. It would be helpfull to anayze the perspective of the people about the candidates

### Text Visualization

1. Text Processing with NLTK
2. Displaying a Word Cloud with nltk and altair

In [1]:
import altair as alt
import pandas as pd
import re
import unicodedata
import requests

### Data

- Data is downloaded from the GitHub link: a story featuring some very interesting people. It doesnt containg emojis, urls, etc.


In [2]:
r = requests.get('https://raw.githubusercontent.com/erickedu85/dataset/refs/heads/master/tweets/tweets_ec_2025.txt')
r.encoding = 'utf-8'

### Data Exploration

It is important to explore how the data is structured, so we can clean it up

In [3]:
lines = r.text.splitlines()
lines[:10]

['@DiegoPonguill10 @DanielNoboaOk @LuisaGonzalezEc JAJAJAAJAJAJAJAJAJAJAJJAJAAJJA okkkkkkk',
 '@hectorjalonm @DanielNoboaOk @LuisaGonzalezEc Ahora vivimos en la miseria antes fuimos el mejor país de latinoamerica..',
 '@Gregori58965636 @yesendiaz @DanielNoboaOk Otro troll basura',
 '@jdiegol2010 @DanielNoboaOk https://t.co/CsLWQdQtnc',
 '@JRamirez2O24 @DanielNoboaOk El tema es respetar a quien eligió el pueblo o no ?',
 '@gladiadorjavier @DanielNoboaOk Yo no lo he visto en las calles',
 '@gladiadorjavier @DanielNoboaOk Pero NOBOA sigue trabajando como Presidente o no?',
 '@Isaac_25_1986 @brillosaaa @DanielNoboaOk Una Dictadura es Dictadura sea la forma que sea, donde se a visto que nombren a una vicepresidenta por decreto que ni el pueblo la eligió, ahí te la dejo.',
 '@JRamirez2O24 @DanielNoboaOk https://t.co/zLgWtMaDIb',
 '@Jorgelr79 @FunerariaAlach @DanielNoboaOk Para mi son culpables al menos de la desaparición forzada, habrá que determinar si son de la ejecución extrajudicial, y e

### Remove Tags

We can see that the lines start tagging someone like @Jorgelr79 @FunerariaAlach @DanielNoboaOk
We need to remove those tags as they are not influencing in the comments of the people

For this case, it is also important to remove the special characters like ,.; and keep only the letters. We will lower all cases to avoid having case issues and remove accents like ó

Also it is important to remove links to other pages

We can also remove strange words that contains jajaj or similar, as well as those ok words. Basically we can remvoe all words that contain only two letters

In [4]:
# We define words that contain certain characters that we want to remove from the text. For example, we might want to remove words that contain "@" (which are usually mentions), "http" (which are usually links), and "#" (which are usually hashtags).
TO_REMOVE = ("@", "http", "#")

# We define a minimum number of characters that a word must have in order to be considered valid. For example, we might want to ignore words that are too short, such as "a", "si", or "asi".
min_characters = 3

In [5]:
def has_minimium_letters(word):
  return word == "noboa" or len(set(word)) > min_characters

def clean_text(words):
  new_line = []
  for word in words:
    if any(token in word for token in TO_REMOVE):
      continue

    text = unicodedata.normalize("NFD", word)
    text = "".join(c for c in text if unicodedata.category(c) != "Mn")
    text = re.sub(r"[^A-Za-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    text = text.lower()
    if has_minimium_letters(text):
      new_line.append(text)
  return ' '.join(new_line)


curated_words = []
for l in lines:
  line = clean_text(l.split(' '))
  if line:
    curated_words.append(line)

In [6]:
curated_words[:10]

['ahora vivimos miseria antes fuimos mejor pais latinoamerica',
 'troll basura',
 'tema respetar quien eligio pueblo',
 'visto calles',
 'pero noboa sigue trabajando presidente',
 'dictadura dictadura forma donde visto nombren vicepresidenta decreto pueblo eligio dejo',
 'culpables menos desaparicion forzada habra determinar ejecucion extrajudicial caso serlo deberian culpable omision pero bajo visto ministro defensa hacen quieren impunidad',
 'presidente tambien mostro completar presidencia necesita pedir licencia porque reeleccion completo presidencia lasso presidencia completa anos',
 'quizas pero ecuador encarcelan matan piense distinto falsean elecciones cosa hace chavismo epoca innombrable chavez comparacion resulta odiosa verdad',
 'dice constitucion asumir cargo']

### Saving Data

We save the curated data to use in future processes

In [7]:
open("curated_words.txt", "w", encoding="utf-8").write('\n'.join(curated_words))

8908823

In [8]:
print("Curated words saved to curated_words.txt")
print("Number of curated lines:", len(curated_words))
print("Number of original lines:", len(lines))
print(f"Percentage of lines retained: {len(curated_words) / len(lines) * 100:.2f}%")

Curated words saved to curated_words.txt
Number of curated lines: 157247
Number of original lines: 195398
Percentage of lines retained: 80.48%
